# Bundesliga predictor

In [55]:
import pandas as pd 
import requests
import numpy as np
from collections import defaultdict
from typing import Dict

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [35]:
def get_table(season: int) -> pd.DataFrame:

    url = f"https://api.openligadb.de/getbltable/bl1/{season}"
    data = requests.get(url).json()

    data

    rows = []
    for team in data:
        rows.append({
            "team_name" : team["teamName"],
            "points" : team["points"],
            "win" : team["won"],
            "draw" : team["draw"],
            "loss" : team["lost"],
            "goals_for" : team["goals"],
            "goals_against" : team["opponentGoals"],
            "goal_diff" : team["goalDiff"]
            })
    table = pd.DataFrame(rows)
    table["position"] = table.index + 1
    return table

In [36]:
get_table(2025)

,team_name,points,win,draw,loss,goals_for,goals_against,goal_diff,position
0,FC Bayern München,63,20,3,1,88,23,65,1
1,Borussia Dortmund,52,15,7,2,51,25,26,2
2,TSG Hoffenheim,46,14,4,6,49,31,18,3
3,VfB Stuttgart,46,14,4,6,48,32,16,4
4,RB Leipzig,44,13,5,6,46,33,13,5
5,Bayer 04 Leverkusen,40,12,4,7,44,29,15,6
6,Eintracht Frankfurt,34,9,7,8,48,49,-1,7
7,SC Freiburg,33,9,6,9,34,39,-5,8
8,FC Augsburg,31,9,4,11,30,41,-11,9
9,1. FC Union Berlin,28,7,7,10,29,38,-9,10


In [3]:
def get_matchday(season: int, matchday: int) -> pd.DataFrame:
    url = f"https://api.openligadb.de/getmatchdata/bl1/{season}/{matchday}"
    data = requests.get(url).json()
    
    rows = []
    for match in data:
        rows.append({
            "matchday": matchday,
            "date": match["matchDateTime"],
            "home": match["team1"]["teamName"],
            "away": match["team2"]["teamName"],
            "score_home": match["matchResults"][-1]["pointsTeam1"] if match["matchResults"] else None,
            "score_away": match["matchResults"][-1]["pointsTeam2"] if match["matchResults"] else None,
        })
    return pd.DataFrame(rows)

# Pull a full season
all_matches = pd.concat([get_matchday(2023, md) for md in range(1, 35)])

In [5]:
all_matches

,matchday,date,home,away,score_home,score_away
0,1,2023-08-18T20:30:00,SV Werder Bremen,FC Bayern München,0,4
1,1,2023-08-19T15:30:00,Bayer 04 Leverkusen,RB Leipzig,3,2
2,1,2023-08-19T15:30:00,VfL Wolfsburg,1. FC Heidenheim 1846,2,0
3,1,2023-08-19T15:30:00,TSG Hoffenheim,SC Freiburg,1,2
4,1,2023-08-19T15:30:00,FC Augsburg,Borussia Mönchengladbach,4,4
...,...,...,...,...,...,...
4,34,2024-05-18T15:30:00,VfL Wolfsburg,1. FSV Mainz 05,1,3
5,34,2024-05-18T15:30:00,TSG Hoffenheim,FC Bayern München,4,2
6,34,2024-05-18T15:30:00,SV Werder Bremen,VfL Bochum,4,1
7,34,2024-05-18T15:30:00,VfB Stuttgart,Borussia Mönchengladbach,4,0


In [ ]:
def create_table(season: int) -> pd.DataFrame:

    season_matches = pd.concat([get_matchday(season, md) for md in range(1, 35)])

    teams: Dict[str, Dict[str, int]] = defaultdict(lambda: {
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goals_for": 0,
            "goals_against": 0,
            "home_wins": 0,
            "home_losses": 0
        })

    for _, row in season_matches.iterrows():
        home = row["home"]
        away = row["away"]

        home_goals = row["score_home"]
        away_goals = row["score_away"]

        teams[home]["goals_for"] += home_goals
        teams[home]["goals_against"] += away_goals
        teams[away]["goals_for"] += away_goals
        teams[away]["goals_against"] += home_goals

        if home_goals > away_goals:
            teams[home]["wins"] += 1
            # teams[home]["home_wins"] += 1
            teams[away]["losses"] += 1
        elif home_goals < away_goals:
            teams[away]["wins"] += 1
            # teams[home]["home_losses"] += 1
            teams[home]["losses"] += 1
        else:
            teams[away]["draws"] += 1
            teams[home]["draws"] += 1

    data = []

    for team, statistics in teams.items():
        goal_diff = statistics["goals_for"] - statistics["goals_against"]
        points = statistics["wins"] * 3 + statistics["draws"] 

        data.append({
            "team_name" : team,
            "points" : points,
            "win" : statistics["wins"],
            "draw" : statistics["draws"],
            "loss" : statistics["losses"],
            "goals_for" : statistics["goals_for"],
            "goals_against" : statistics["goals_against"],
            "goal_diff" : goal_diff,
            # "home_wins" : statistics["home_wins"],
            # "home_losses" : statistics["home_losses"]
        })

    table = pd.DataFrame(data)

    table = table.sort_values(["points", "goal_diff", "goals_for"], ascending= [False, False, False]).reset_index(drop= True)

    table["position"] = table.index + 1

    return table

In [21]:
create_table(2022)

,team_name,points,win,draw,loss,goals_for,goals_against,goal_diff,home_wins,home_losses,position
0,FC Bayern München,84,26,6,2,57,10,47,14,2,1
1,Borussia Dortmund,62,18,8,8,43,22,21,12,2,2
2,RB Leipzig,56,16,8,10,29,18,11,11,2,3
3,SC Freiburg,50,11,17,6,19,14,5,7,2,4
4,Eintracht Frankfurt,50,13,11,10,29,29,0,9,3,5
5,Borussia Mönchengladbach,49,12,13,9,23,28,-5,8,2,6
6,VfL Wolfsburg,46,11,13,10,30,25,5,6,5,7
7,1. FSV Mainz 05,46,12,10,12,25,27,-2,7,5,8
8,Bayer 04 Leverkusen,44,11,11,12,23,21,2,8,5,9
9,1. FC Union Berlin,43,9,16,9,20,21,-1,6,2,10


It turns out the data correct for 2023/24 and 2024/25 seasons, but is incorrect for seasons prior. I will directly fetch the standings data, but will not be able to compute home wins and away wins ... Hopefully it doesn't affect model performance too much.

In [53]:
def prep_data(start_date : int, end_date : int): #end date should be 2024
    season_tables : Dict[str, pd.DataFrame] = {}
    for year in range(start_date, end_date + 1):
        table = get_table(year)
        season_tables[f"{year}"] = table
    feature_rows = []
    target_rows = []
    for i in range(start_date, end_date):
        prev_table = season_tables[f"{i}"].copy().set_index("team_name")
        curr_table = season_tables[f"{i +1}"].copy().set_index("team_name")
        relegated = prev_table.tail(3)
        promoted_stats = relegated.mean().to_dict()

    for team, row in curr_table.iterrows():
        if team in prev_table.index:
            features = prev_table.loc[team][["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]].to_dict()
        else:
            features = {k : promoted_stats[k] for k in ["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]}
        feature_rows.append(features)
        target_rows.append(row["position"])
    X_train = pd.DataFrame(feature_rows)
    y_train = pd.Series(target_rows)

    last_feature_rows = []
    last_table = season_tables[f"{end_date}"].copy().set_index("team_name")
    last_relegated = last_table.tail(2)
    last_promoted_stats = last_relegated.mean().to_dict()
    last_teams = last_table.index.tolist()

    last_promoted_teams = ["FC Köln", "Hamburger SV"]
    last_relegated_teams = last_relegated.index.tolist()

    for team in last_teams:
        if team in last_relegated_teams:
            continue
        features = last_table.loc[team][["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]].to_dict()
        last_feature_rows.append((team, features))
    for team in last_promoted_teams:
        if team not in last_teams:
            feats = {k : last_promoted_stats[k] for k in ["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]}
            last_feature_rows.append((team, feats))
    latest_features_df = pd.DataFrame([feats for _, feats in last_feature_rows],
                                      index=[t for t, _ in last_feature_rows])
    return X_train, y_train, latest_features_df

Now, we start the random forest using Sklearn.

In [ ]:
def train_model(X, y) -> Pipeline: 
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            random_state=42,
            class_weight="balanced"
        ))
    ])

    model.fit(X, y)
    return model